# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Building the Global Development Monitor

### Scenario

You've just joined the analytics team at the **Global Development Observatory**, an NGO
that briefs policymakers on world development trends. Your director has been burned
before by bad visualizations, and gives you a blunt brief:

1. *"The last analyst showed me a map that made me think Greenland mattered more than
   India. Don't let that happen again."*
2. *"We're about to publish a model predicting life expectancy. If a journalist asks
   'why did the model predict THAT for Norway specifically,' I need an answer, not just
   a feature-importance bar chart for the whole model."*
3. *"Every number in our reports is an estimate. I want our numbers to LOOK like
   estimates, not like certainties."*
4. *"I want one screen I can glance at every morning that tells me what's changing."*

You'll work through the same steps a real analyst would: fix a misleading map, explain
individual model predictions, add honest uncertainty to an estimate and a trend, then
build a small coordinated-views dashboard.

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection**
are for you to answer in your own words.

## Part 0 — Setup (given)

Run this cell as-is. It loads the real dataset and real country centroids you'll be
working with.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
import shap

import ipywidgets as widgets
from ipywidgets import interact, Dropdown

np.random.seed(0)

gapminder = px.data.gapminder()
centroids = pd.read_csv("country_centroids_computed.csv")
gap_latest = gapminder[gapminder.year == 2007].merge(centroids, on="iso_alpha", how="left")
gap_latest = gap_latest.assign(total_gdp=gap_latest["pop"] * gap_latest.gdpPercap)

print(f"Loaded {gapminder.country.nunique()} countries, years {sorted(gapminder.year.unique())}.")
gap_latest[["country", "continent", "year", "lifeExp", "pop", "gdpPercap"]].head()

## Task 1 — Fix the Choropleth Trap (Complaint #1)

**Real-world usage:** The director was misled by a raw-count choropleth that made a
huge, sparsely-populated region look more important than it should.

**Why this technique:** Distinguishing **counts** from **rates** is the single most
important geospatial fix from the lecture — a "count" choropleth is dominated by
population size, not by the thing you actually care about.

**TODO:**
1. Build a choropleth of `total_gdp` (a count-like aggregate) using `px.choropleth`.
2. Build a second choropleth of `gdpPercap` (the rate).
3. Print the top-5 countries by each, and compare.

In [ ]:
# TODO 1: choropleth of total_gdp
fig1 = px.choropleth(
    gap_latest,
    locations="iso_alpha",
    color="total_gdp",
    hover_name="country",
    color_continuous_scale="Reds",
    title="World Total GDP (2007) - Raw Aggregate (Dominated by Population Size)"
)
fig1.show()

# TODO 2: choropleth of gdpPercap
fig2 = px.choropleth(
    gap_latest,
    locations="iso_alpha",
    color="gdpPercap",
    hover_name="country",
    color_continuous_scale="Viridis",
    title="World GDP per Capita (2007) - Rate/Intensity (Measures Economic Living Standard)"
)
fig2.show()

# TODO 3: top 5 countries by each -- print both lists
top_total = gap_latest.sort_values("total_gdp", ascending=False)[["country", "total_gdp"]].head(5)
top_rate = gap_latest.sort_values("gdpPercap", ascending=False)[["country", "gdpPercap"]].head(5)
print("Top 5 by total GDP:\n", top_total.to_string(index=False))
print("\nTop 5 by GDP per capita:\n", top_rate.to_string(index=False))


**Reflection:** Which countries appear in one top-5 list but not the other? If you had
to pick ONE map to show the director, which would you pick, and why?

_Your answer:_
The total GDP top-5 is dominated by populous superpowers (United States, China, Japan, Germany, United Kingdom), whereas the GDP per capita top-5 features smaller, highly productive economies (Norway, Kuwait, Singapore, United States, Ireland), with China and India disappearing entirely from the rate list. If we must pick ONE map to brief the director, we must choose **GDP per capita (the rate)**. Showing raw total GDP falls into the "Choropleth Trap" where land area and total population confound the metric, misleading viewers into mistaking massive national population counts for high living standards and individual prosperity.


## Task 2 — Explain One Specific Prediction (Complaint #2)

**Real-world usage:** The director needs to answer "why did the model predict THAT for
Norway specifically" — a global importance bar chart cannot answer a question about one
country.

**Why this technique:** SHAP local importance decomposes ONE prediction into
per-feature, signed contributions — exactly what a journalist question needs.

**TODO:**
1. Fit a `RandomForestRegressor` predicting `lifeExp` from `["gdpPercap", "pop", "year"]`
   on the full `gapminder` dataset.
2. Build a `shap.TreeExplainer` and compute `shap_values` for all rows.
3. Find the row index for Norway, 2007, and plot its 3 SHAP values as a diverging
   horizontal bar chart (blue if positive, red if negative).

In [ ]:
X_model = gapminder[["gdpPercap", "pop", "year"]].values
y_model = gapminder["lifeExp"].values

# TODO 1: fit the model
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=0)
rf.fit(X_model, y_model)

# TODO 2: SHAP explainer + values
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_model)

# TODO 3: find Norway's row, plot its SHAP contributions
row_idx = gapminder[(gapminder.country == "Norway") & (gapminder.year == 2007)].index[0]
vals = shap_values[row_idx]
feature_names = ["gdpPercap", "pop", "year"]

fig, ax = plt.subplots(figsize=(7, 3.5))
# plot vals as a diverging horizontal bar chart here
bar_colors = ["#3b6ea5" if v >= 0 else "#b5432e" for v in vals]
ax.barh(feature_names, vals, color=bar_colors)
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Norway (2007): Local Feature Contributions to Predicted Life Expectancy", fontsize=11, fontweight="bold")
ax.set_xlabel("SHAP Value (Impact on Life Expectancy in years)")
for i, v in enumerate(vals):
    ax.text(v + (0.2 if v >= 0 else -0.2), i, f"{v:+.2f} yrs", va="center", ha="left" if v >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.show()


**Reflection:** Which feature contributed most POSITIVELY to Norway's predicted life
expectancy, and which (if any) contributed negatively? Is this the same answer you'd get
from the model's GLOBAL feature importance alone?

_Your answer:_
`gdpPercap` contributed most strongly positive to Norway's predicted life expectancy (adding over 7+ years above the expected baseline due to Norway's extraordinarily high income level), while `pop` had a small negative contribution. This specific signed insight cannot be derived from GLOBAL feature importance alone: global importance only indicates that `gdpPercap` is the most influential feature on average across all world observations (unsigned magnitude), but cannot tell a journalist whether `pop` or `year` pushed Norway's specific prediction up or down.


## Task 3 — Make an Estimate Look Like an Estimate (Complaint #3)

**Real-world usage:** Every number in the report should visually communicate its own
uncertainty, not look like a fixed fact.

**Why this technique:** Bootstrap confidence intervals + error bars turn a bare point
estimate into an honest range.

**TODO:**
1. Write `bootstrap_ci(values, n_boot=2000)` that resamples `values` with replacement
   `n_boot` times, computes the mean each time, and returns `(mean, ci_low, ci_high)`
   using the 2.5th and 97.5th percentiles of the bootstrap means.
2. Compute a 95% CI for mean `lifeExp` in 2007 for **two** continents of your choice.
3. Plot both as error bars.

In [ ]:
def bootstrap_ci(values, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    # TODO 1: resample `values` with replacement n_boot times, take the mean each time,
    # then return (overall mean, 2.5th percentile of boot means, 97.5th percentile)
    boot_means = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)]
    overall_mean = float(np.mean(values))
    ci_low = float(np.percentile(boot_means, 2.5))
    ci_high = float(np.percentile(boot_means, 97.5))
    return overall_mean, ci_low, ci_high

# TODO 2: compute CIs for two continents of your choice
continent_a, continent_b = "Africa", "Europe"  # feel free to change these
values_a = gap_latest[gap_latest.continent == continent_a].lifeExp.values
values_b = gap_latest[gap_latest.continent == continent_b].lifeExp.values

result_a = bootstrap_ci(values_a)
result_b = bootstrap_ci(values_b)

# TODO 3: plot both as error bars (mean +/- CI half-width) on one chart
continents = [continent_a, continent_b]
means = [result_a[0], result_b[0]]
yerr = [
    [result_a[0] - result_a[1], result_b[0] - result_b[1]],
    [result_a[2] - result_a[0], result_b[2] - result_b[0]]
]

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.errorbar(continents, means, yerr=yerr, fmt="o", color="#3b6ea5", ecolor="#b5432e", elinewidth=2, capsize=6, capthick=2, markersize=8)
ax.set_ylabel("Mean Life Expectancy (2007)")
ax.set_title("95% Bootstrap Confidence Intervals for Mean Life Expectancy")
for i, (m, res) in enumerate(zip(means, [result_a, result_b])):
    ax.text(i + 0.08, m, f"Mean: {m:.1f}\n[95% CI: {res[1]:.1f}, {res[2]:.1f}]", va="center", fontsize=9)
plt.tight_layout()
plt.show()


**Reflection:** Which of your two continents has a WIDER confidence interval? Based on
the lecture, is that because of higher variation among countries (SD), a smaller sample
size (n), or both?

_Your answer:_
**Africa** has a visibly wider confidence interval than Europe. Based on sampling distribution theory ($\text{SE} \approx \frac{\sigma}{\sqrt{n}}$), this is driven primarily by **much higher variation among countries (standard deviation $\sigma$)**: life expectancy in Africa in 2007 spans widely from under 40 years in nations hit by conflict/epidemics to over 70 years in Northern and island nations, whereas European nations have uniformly high life expectancies with low variance, yielding a tight confidence interval despite Europe having fewer countries than Africa in this dataset.


## Task 4 — Build the Morning Dashboard (Complaint #4)

**Real-world usage:** The director wants one screen to glance at every morning.

**Why this technique:** Coordinated Multiple Views combine complementary chart types
(map = where, trend = how changing, bar = which is biggest) so several monitoring
questions are answered from a single glance.

**TODO:** Build a 1x3 dashboard:
1. A map-style scatter of country centroids, colored by `lifeExp`.
2. A line chart of world-average `lifeExp` by year (use `gapminder.groupby("year")`).
3. A horizontal bar chart of total population by continent in 2007, sorted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# TODO 1: map-style scatter (axes[0])
sc = axes[0].scatter(gap_latest.centroid_lon, gap_latest.centroid_lat, c=gap_latest.lifeExp, cmap="viridis", s=30, alpha=0.8, edgecolors="none")
axes[0].set_title("Global Centroids: Life Expectancy (2007)", fontweight="bold")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
plt.colorbar(sc, ax=axes[0], orientation="horizontal", pad=0.2, label="Life Expectancy (years)")

# TODO 2: world trend line (axes[1])
world_trend = gapminder.groupby("year")["lifeExp"].mean().reset_index()
axes[1].plot(world_trend["year"], world_trend["lifeExp"], marker="o", color="#2b5c8f", linewidth=2)
axes[1].set_title("World Average Life Expectancy Trend", fontweight="bold")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Average Life Expectancy")
axes[1].grid(True, linestyle="--", alpha=0.6)

# TODO 3: population by continent bar chart (axes[2])
pop_by_cont = gap_latest.groupby("continent")["pop"].sum().sort_values(ascending=True) / 1e6
axes[2].barh(pop_by_cont.index, pop_by_cont.values, color="#55A868")
axes[2].set_title("Total Population by Continent (2007)", fontweight="bold")
axes[2].set_xlabel("Population (Millions)")
axes[2].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


**Reflection:** If the director could only keep ONE of your three panels for a
5-second morning glance, which would you keep, and why? What specific monitoring
question (per the lecture's "exceptions / trends / emerging patterns") does it answer
best?

_Your answer:_
For a 5-second morning glance, the director should keep **Panel 1 (the geographic centroid scatter map)**. According to the monitoring dashboard taxonomy from the lecture, operational monitoring prioritizes spotting *exceptions and emerging geographic hotspots* before examining macro aggregates. The centroid map color-codes every country simultaneously, allowing pre-attentive detection of geographic anomalies or localized crises across the globe that would be completely hidden by smooth world-average trend lines or aggregated continent bars.


## Task 5 — The Decision (No Code)

**Reflection (final):** Write a 4–6 sentence recommendation to your director covering
the whole notebook. Your answer should:
- State which map framing (count vs. rate) you'd standardize on for all future reports,
  and why.
- State whether you'd lead with global or local feature importance when a journalist
  asks about a specific country, citing the lecture's global-vs-local distinction.
- Name one thing in your dashboard that should NEVER be shown as a bare point estimate
  without uncertainty, and why.

_Your final recommendation here:_
We recommend standardizing all future NGO briefings on **rate-based choropleth framing** (such as GDP per capita or infant mortality per 1,000 births) rather than raw counts, ensuring that sheer geographic landmass and population size do not mislead policymakers. When responding to journalist inquiries regarding a specific country's forecast, we must lead with **local feature importance (SHAP)**; global importance describes average model mechanics across the entire world, whereas local SHAP isolates the exact positive and negative drivers unique to that individual nation. Finally, **regional mean projections and future trend estimates should never be displayed as bare point estimates without confidence intervals or uncertainty bands**, because point estimates convey an illusion of certainty and conceal critical sampling variance and geopolitical unpredictability.
